# Pytest fixtures coverage

<div style="display:flex;justify-content:space-between;align-items:center;background:#0d0d0f;border:1px solid #1e1e24;border-left:4px solid #4fc3f7;border-radius:6px;padding:clamp(1rem,2.5vw,1.8rem) clamp(1.2rem,3vw,2.4rem);margin-bottom:2rem;position:relative;overflow:hidden;box-shadow:0 4px 32px rgba(0,0,0,0.5);font-family:'Segoe UI',sans-serif;">
<div style="position:absolute;top:0;left:0;right:0;bottom:0;background:radial-gradient(ellipse at 0% 50%,rgba(79,195,247,0.07) 0%,transparent 60%);pointer-events:none;"></div>
<div style="display:flex;flex-direction:column;gap:0.3rem;">
<p style="font-size:clamp(1.3rem,3.5vw,2.4rem);color:#f0f0f5;margin:0;line-height:1.1;font-weight:700;letter-spacing:-0.01em;">pytest: Fixtures, Parametrisierung und Coverage</p>
<p style="font-size:clamp(0.75rem,1.6vw,1rem);color:#7a7a90;margin:0;letter-spacing:0.04em;font-weight:300;">Development Expert Python / PCAP &nbsp;|&nbsp; Kapitel 10: Testing &nbsp;|&nbsp; Notebook 10c</p>
</div>
</div>

**Legende**

> **[Kursinhalt]** Dieses Notebook ist kein PCAP-Pruefungsinhalt

---

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
1. Das naechste Problem: Wiederholung in Tests
</span>
</div>

In 10b haben wir 30 Tests geschrieben. Das funktioniert gut -- aber schau dir die Tests fuer `heilen()` an: Jeder Test erstellt dieselben Grundannahmen neu. Und wenn wir anfangen, komplexere Objekte zu testen -- zum Beispiel einen ganzen `Held` mit Name, HP, Angriff und Level -- dann wuerde jeder Test dieses Objekt neu aufbauen.

Schauen wir uns an wie das aussieht wenn man es nicht loest:

In [ ]:
# Stell dir vor, wir testen eine Held-Klasse
# Ohne Fixtures sieht jeder Test so aus:

beispiel_ohne_fixture = '''
def test_held_erhaelt_schaden():
    held = Held(name="Aldric", hp=100, max_hp=100, angriff=15, verteidigung=5)
    held.erleidet_schaden(20)
    assert held.hp == 80

def test_held_wird_geheilt():
    held = Held(name="Aldric", hp=100, max_hp=100, angriff=15, verteidigung=5)
    held.erleidet_schaden(50)
    held.heilen(20)
    assert held.hp == 70

def test_held_ist_besiegt():
    held = Held(name="Aldric", hp=100, max_hp=100, angriff=15, verteidigung=5)
    held.erleidet_schaden(100)
    assert held.ist_besiegt() == True

# ... 20 weitere Tests, alle beginnen mit denselben 5 Zeilen
'''
print('Problem: Jeder Test baut denselben Held neu auf.')
print('Wenn sich die Held-Klasse aendert, muss man 20 Tests anpassen.')

Das verletzt das DRY-Prinzip (Don't Repeat Yourself) -- und macht Tests schwer zu warten. pytest loest das mit **Fixtures**.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
2. Fixtures -- Testdaten einmal definieren, ueberall verwenden
</span>
</div>

Eine **Fixture** ist eine Funktion die mit `@pytest.fixture` dekoriert wird. pytest ruft sie automatisch auf und uebergibt ihr Ergebnis an den Test -- einfach indem man den Fixture-Namen als Parameter des Tests angibt.

Das klingt wie Magie, ist aber simples Dependency Injection: pytest sieht dass ein Test einen Parameter mit dem Namen `standard_held` hat, sucht nach einer Fixture dieses Namens und reicht den Rueckgabewert weiter.

In [ ]:
# Wir erweitern combat.py um eine Held-Klasse
held_klasse = '''
from combat import berechne_schaden, heilen as heilen_fn, ist_besiegt, statuseffekt_anwenden


class Held:
    def __init__(self, name, hp, max_hp, angriff, verteidigung):
        self.name        = name
        self.hp          = hp
        self.max_hp      = max_hp
        self.angriff     = angriff
        self.verteidigung = verteidigung

    def erleidet_schaden(self, schaden):
        netto = berechne_schaden(schaden, self.verteidigung)
        self.hp -= netto

    def heilen(self, menge):
        self.hp = heilen_fn(self.hp, self.max_hp, menge)

    def ist_besiegt(self):
        return ist_besiegt(self.hp)

    def __repr__(self):
        return f"Held({self.name}, HP={self.hp}/{self.max_hp})"
'''

with open('held.py', 'w') as f:
    f.write(held_klasse)
print('held.py erstellt')

In [ ]:
test_mit_fixtures = '''
import pytest
from held import Held


# Fixture: wird vor jedem Test der sie verwendet aufgerufen
@pytest.fixture
def standard_held():
    """Ein frischer Held mit vollen HP fuer jeden Test."""
    return Held(name="Aldric", hp=100, max_hp=100, angriff=15, verteidigung=5)


@pytest.fixture
def verletzter_held():
    """Ein Held der bereits Schaden erlitten hat."""
    return Held(name="Lyria", hp=30, max_hp=100, angriff=12, verteidigung=3)


# Tests verwenden Fixture als Parameter -- pytest injiziert sie automatisch
def test_held_erhaelt_schaden(standard_held):
    standard_held.erleidet_schaden(20)
    # Nettoschaden: 20 - 5 (Verteidigung) = 15
    assert standard_held.hp == 85


def test_held_wird_geheilt(verletzter_held):
    verletzter_held.heilen(20)
    assert verletzter_held.hp == 50


def test_held_heilung_cap(verletzter_held):
    # Heilung weit ueber max_hp -- wird gekappt
    verletzter_held.heilen(999)
    assert verletzter_held.hp == 100


def test_held_ist_besiegt(standard_held):
    standard_held.erleidet_schaden(999)
    assert standard_held.ist_besiegt() == True


def test_held_lebt_noch(standard_held):
    standard_held.erleidet_schaden(10)
    assert standard_held.ist_besiegt() == False


def test_jeder_test_bekommt_frischen_held(standard_held):
    # Dieser Test aendert den Held -- aber der naechste Test bekommt
    # einen frischen Held, nicht diesen veraenderten
    standard_held.erleidet_schaden(999)
    assert standard_held.hp < 0
'''

with open('test_held.py', 'w') as f:
    f.write(test_mit_fixtures)
print('test_held.py erstellt')

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'test_held.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

Wichtig: pytest erstellt die Fixture **vor jedem Test neu**. `test_held_ist_besiegt` bekommt einen Held mit 100 HP -- nicht den Held mit 0 HP aus `test_held_erhaelt_schaden`. Jeder Test ist isoliert. Das ist das Kernprinzip.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
3. Parametrisierung -- einen Test mit vielen Eingaben
</span>
</div>

Schau dir die Tests fuer `berechne_schaden()` aus 10b an -- viele Tests die dasselbe pruefen, nur mit anderen Zahlen. Das ist ein weiteres Wiederholungsmuster.

`@pytest.mark.parametrize` loest das: Man schreibt den Test einmal und gibt eine Liste von Eingabe-Erwartungs-Paaren mit. pytest fuehrt den Test automatisch fuer jede Kombination aus -- und zeigt jeden Fall einzeln in der Ausgabe.

In [ ]:
test_parametrize = '''
import pytest
from combat import berechne_schaden, heilen, trefferchance, statuseffekt_anwenden


# Ohne Parametrisierung: 4 separate Tests
def test_schaden_v1(): assert berechne_schaden(50, 20) == 30
def test_schaden_v2(): assert berechne_schaden(20, 20) == 1
def test_schaden_v3(): assert berechne_schaden(10, 50) == 1
def test_schaden_v4(): assert berechne_schaden(100, 0) == 100


# Mit Parametrisierung: ein Test, vier Faelle
# Format: "angriff, verteidigung, erwartet"
@pytest.mark.parametrize("angriff, verteidigung, erwartet", [
    (50, 20, 30),   # Normalfall
    (20, 20,  1),   # Grenzwert: gleiche Werte -> Mindestschaden
    (10, 50,  1),   # Grenzwert: Verteidigung > Angriff -> Mindestschaden
    (100, 0, 100),  # Grenzwert: keine Verteidigung
    (1,   0,   1),  # Grenzwert: minimaler Angriff
])
def test_berechne_schaden(angriff, verteidigung, erwartet):
    assert berechne_schaden(angriff, verteidigung) == erwartet


# Grenzwerte fuer trefferchance
@pytest.mark.parametrize("angreifer, ziel, erwartet", [
    (10, 10,  50),   # gleich schnell
    (10,  8,  60),   # etwas schneller
    ( 1, 20,  10),   # Grenzwert: Minimum -- sehr langsam
    (20,  1,  95),   # Grenzwert: Maximum -- sehr schnell
    ( 2, 20,  10),   # Aequivalenzklasse: unter Minimum bleibt bei 10
    (19,  1,  95),   # Aequivalenzklasse: ueber Maximum bleibt bei 95
])
def test_trefferchance(angreifer, ziel, erwartet):
    assert trefferchance(angreifer, ziel) == erwartet


# Alle gueltigen Statuseffekte
@pytest.mark.parametrize("effekt, schaden, erwartet", [
    ("normal",    20, 20),
    ("vergiftet", 20, 30),
    ("geschwaeht",20, 15),
    ("verstaerkt",20, 40),
    ("vergiftet",  0,  0),   # Grenzwert: Schaden = 0
])
def test_statuseffekt_anwenden(effekt, schaden, erwartet):
    assert statuseffekt_anwenden(schaden, effekt) == erwartet
'''

with open('test_parametrize.py', 'w') as f:
    f.write(test_parametrize)
print('test_parametrize.py erstellt')

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'test_parametrize.py', '-v'],
    capture_output=True, text=True
)
print(result.stdout)

pytest zeigt jeden Testfall einzeln -- mit den Parameterwerten im Namen:

```
test_berechne_schaden[50-20-30] PASSED
test_berechne_schaden[20-20-1]  PASSED
test_berechne_schaden[10-50-1]  PASSED
```

Wenn ein einzelner Fall fehlschlaegt, sieht man sofort welche Parameterkombination das Problem verursacht hat.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
4. Coverage -- wie viel Code ist wirklich getestet?
</span>
</div>

Wir haben viele Tests geschrieben -- aber woher wissen wir ob wir alle Codepfade abgedeckt haben? Gibt es Zeilen in `combat.py` die nie ausgefuehrt werden weil kein Test sie erreicht?

**Coverage** (Testabdeckung) misst genau das: welcher Prozentsatz des Codes durch Tests ausgefuehrt wird. Das Werkzeug dafuer heisst `pytest-cov`.

```bash
pip install pytest-cov
```

In [ ]:
try:
    import pytest_cov
    print('pytest-cov installiert')
except ImportError:
    print('Bitte ausfuehren: pip install pytest-cov')

In [ ]:
import subprocess

# --cov=combat: Coverage fuer combat.py messen
# --cov-report=term-missing: zeigt welche Zeilen fehlen
result = subprocess.run(
    ['python', '-m', 'pytest', 'test_combat.py',
     '--cov=combat', '--cov-report=term-missing', '-v'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0 and 'no module named pytest_cov' in result.stdout.lower():
    print('pytest-cov nicht installiert -- pip install pytest-cov')

Die Coverage-Ausgabe sieht so aus:

```
Name        Stmts   Miss  Cover   Missing
-------------------------------------------
combat.py      28      0   100%
```

100% bedeutet: jede Zeile in `combat.py` wurde durch mindestens einen Test ausgefuehrt. Wenn eine Zeile nie erreicht wird, erscheint sie in der `Missing`-Spalte -- ein Hinweis dass entweder ein Testfall fehlt oder der Code selbst unerreichbar ist.

**Wichtig:** 100% Coverage bedeutet nicht dass der Code fehlerfrei ist. Es bedeutet nur dass jede Zeile ausgefuehrt wurde -- nicht dass alle moeglichen Eingaben getestet wurden. Coverage ist ein Werkzeug, keine Garantie.

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
5. Das AAA-Prinzip -- wie ein guter Test aussieht
</span>
</div>

Ein guter Test folgt dem **AAA-Prinzip**: Arrange, Act, Assert.

- **Arrange** -- Vorbereitung: Testdaten und Ausgangszustand herstellen
- **Act** -- Ausfuehren: genau eine Aktion ausfuehren
- **Assert** -- Pruefen: das Ergebnis pruefen

Das klingt trivial -- aber es zwingt dazu, jeden Test auf genau eine Sache zu fokussieren. Ein Test der viele Dinge gleichzeitig prueft ist schwer zu lesen und wenn er fehlschlaegt, weiss man nicht was genau das Problem war.

In [ ]:
# AAA-Prinzip am Beispiel

# SCHLECHT: prueft zu viele Dinge auf einmal
def test_kampfrunde_komplett_schlecht():
    from held import Held
    from Aetheria_Game.combat import berechne_schaden
    held    = Held("Aldric", 100, 100, 15, 5)
    gegner  = Held("Drache", 200, 200, 30, 10)
    # Held greift an
    schaden_an_gegner = berechne_schaden(held.angriff, gegner.verteidigung)
    gegner.erleidet_schaden(held.angriff)
    # Gegner greift an
    schaden_an_held = berechne_schaden(gegner.angriff, held.verteidigung)
    held.erleidet_schaden(gegner.angriff)
    # Alles auf einmal pruefen -- wenn etwas fehlschlaegt, was genau?
    assert schaden_an_gegner == 5
    assert gegner.hp == 195
    assert schaden_an_held == 25
    assert held.hp == 75
    assert held.ist_besiegt() == False
    assert gegner.ist_besiegt() == False
    print('Dieser Test prueft 6 Dinge -- zu viel')


# GUT: jeder Test prueft genau eine Sache
from held import Held
import pytest

@pytest.fixture
def held_und_gegner():
    # Arrange: Testdaten vorbereiten
    held   = Held("Aldric", 100, 100, 15, 5)
    gegner = Held("Drache", 200, 200, 30, 10)
    return held, gegner

def test_held_schaden_an_gegner(held_und_gegner):
    held, gegner = held_und_gegner
    # Act
    gegner.erleidet_schaden(held.angriff)
    # Assert -- genau eine Sache
    assert gegner.hp == 195   # 200 - (15-10) = 195

def test_gegner_schaden_an_held(held_und_gegner):
    held, gegner = held_und_gegner
    # Act
    held.erleidet_schaden(gegner.angriff)
    # Assert
    assert held.hp == 75   # 100 - (30-5) = 75

print('Zwei Tests, jeder prueft genau eine Sache')

<div style="background:#0b1929;border-left:4px solid #4fc3f7;border-radius:4px;padding:0.75em 1.2em;margin:2rem 0 1.2rem 0;font-family:'Segoe UI',sans-serif;">
<span style="font-size:clamp(1rem,2.2vw,1.25rem);font-weight:600;color:#e8eaf6;letter-spacing:-0.01em;">
<span style="font-size:0.65rem;letter-spacing:0.1em;text-transform:uppercase;color:#4fc3f7;background:rgba(79,195,247,0.12);border:1px solid rgba(79,195,247,0.3);border-radius:3px;padding:0.2em 0.55em;margin-right:0.75em;vertical-align:middle;white-space:nowrap;">Kursinhalt</span>
6. Projektstruktur und conftest.py
</span>
</div>

In einem echten Projekt liegen Tests in einem eigenen Ordner. Fixtures die mehrere Testdateien brauchen, kommen in eine spezielle Datei: `conftest.py`. pytest findet und laedt sie automatisch.

```
mein_projekt/
    combat.py
    held.py
    tests/
        conftest.py          <-- gemeinsame Fixtures fuer alle Tests
        test_combat.py
        test_held.py
        test_integration.py
```

`conftest.py` enthaelt keine Tests -- nur Fixtures:

In [ ]:
import os
os.makedirs('tests', exist_ok=True)

conftest = '''
import pytest
from held import Held


@pytest.fixture
def standard_held():
    """Frischer Held mit vollen HP -- verfuegbar in allen Testdateien."""
    return Held(name="Aldric", hp=100, max_hp=100, angriff=15, verteidigung=5)


@pytest.fixture
def verletzter_held():
    """Held mit niedrigen HP -- fuer Tests rund um Heilung und Tod."""
    return Held(name="Lyria", hp=10, max_hp=100, angriff=12, verteidigung=3)


@pytest.fixture
def starker_gegner():
    """Ein starker Gegner fuer Kampftests."""
    return Held(name="Drache", hp=500, max_hp=500, angriff=60, verteidigung=20)
'''

with open('tests/conftest.py', 'w') as f:
    f.write(conftest)
print('tests/conftest.py erstellt')
print('Alle Tests in tests/ haben jetzt Zugriff auf standard_held, verletzter_held, starker_gegner')

---

**Zusammenfassung**

| Konzept | Erklaerung |
|---------|------------|
| `@pytest.fixture` | Testdaten einmal definieren -- pytest injiziert sie automatisch |
| Fixture ist isoliert | Jeder Test bekommt eine frische Kopie -- Aenderungen wirken sich nicht auf andere Tests aus |
| `@pytest.mark.parametrize` | Einen Test mit vielen Eingabe-Erwartungs-Paaren laufen lassen |
| Coverage | Misst wie viel Prozent des Codes durch Tests ausgefuehrt wird |
| `--cov --cov-report=term-missing` | Zeigt welche Zeilen nicht getestet wurden |
| AAA-Prinzip | Arrange, Act, Assert -- ein Test prueft genau eine Sache |
| `conftest.py` | Gemeinsame Fixtures fuer alle Testdateien im Verzeichnis |

---

**Weiter:** In Notebook **10d** automatisieren wir das Ausfuehren der Tests mit **GitHub Actions** -- damit bei jedem Push in das Repository alle Tests automatisch laufen.